# Employee-Attrition-Analysis-HR-Strategy-Proposal

架空企業を対象にした教育用ケーススタディです。実行前にREADMEのデータ利用条件と限界を確認してください。

公開版は試行モデルまでを含みます。PDFのAUC 0.8486と施策シミュレーションの最終コードは未収録です。修正前の保存出力はREADMEの参照コミットにあります。以下の出力は再実行時に生成されます。


データセットの内容
・架空のIT企業を対象とした教育用の提供データ。配布元URL・利用許諾は未確認。実在企業の実測データとして扱わない。

・約1500人の従業員についてのcsvファイルが提供されている

Education

 'Below College'

 'College'

 'Bachelor'

 'Master'

 'Doctor'


EnvironmentSatisfaction

 'Low'

 'Medium'

 'High'

 'Very High'


JobInvolvement  

 'Low'

 'Medium'

 'High'

 'Very High'


JobSatisfaction  

 'Low'

 'Medium'

 'High'

 'Very High'


RelationshipSatisfaction  

 'Low'

 'Medium'

 'High'

 'Very High'


WorkLifeBalance  

 'Bad'

 'Good'

 'Better'

 'Best'


RemoteWork

元の説明は on-site work / full remote ですが、保存出力には0〜5の値があります。符号と区分の対応は提供元への確認が必要です。


Incentive : rewards for performance

StressRating：Stress level evaluation by supervisors and HR, including occupational physician assessments.
1.  ‘Very Low’
3.  ‘Average’
5.  ‘Very High’


StressSelfReported：Self-reported stress level obtained from internal company stress checks.
1.  ‘Very Low’
3.  ‘Average’
5.  ‘Very High’


WelfareBenefits：Level of overall welfare benefits utilization frequency.
1.  ‘Rarely Used’
4.  ‘Very Frequently Used’


InHouseFacility：Whether an employee utilizes in-house facilities (e.g., gym, café, cafeteria) above a certain threshold.
0.  ‘No’
1.  ‘Yes’


ExternalFacility：Whether an employee utilizes external facilities (e.g., company-affiliated facilities, language schools) above a certain threshold.
0.  ‘No’
1.  ‘Yes’


ExtendedLeave：Whether an employee has taken extended leave (e.g., parental leave, maternity leave, refreshment leave, volunteer leave) for a certain period.
0.  ‘No’
1.  ‘Yes’


FlexibleWork：Whether an employee utilizes flexible working arrangements (e.g., flextime, reduced working hours) above a certain threshold.
0.  ‘No’
1.  ‘Yes’


PerformanceIndex：An index evaluating an employee's labor productivity on a 100-point score scale.

PerformanceRating：Employee performance score determined by the annual HR evaluation.

 'Low'

 'Good'

 'Excellent'

 'Outstanding'


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pandas import DataFrame, Series
import math
import japanize_matplotlib


In [ ]:
# ローカルJupyter / Colab共通。Google Driveのマウントは不要です。
from pathlib import Path
import os
DATA_PATH = Path(os.environ.get("ATTRITION_DATA_PATH", "data/data.csv"))


In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"{DATA_PATH} がありません。READMEに従い利用権限のある提供CSVを配置してください。")
df = pd.read_csv(DATA_PATH)
# EDAで作る区間列をモデル入力に混入させないため原データを保存
raw_df = df.copy()


In [ ]:
print(df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

Attrition:離職

In [ ]:
df['Incentive'].unique()

In [ ]:
df['MonthlyIncome'].unique()

In [ ]:
df['PerformanceIndex'].unique()

In [ ]:
df['PerformanceRating'].unique()

In [ ]:

import math
categorical_cols = [
    'BusinessTravel', 'Department', 'EducationField', 'Gender',
    'JobRole', 'MaritalStatus', 'OverTime', 'HowToEmploy'
]


n_cols = 2
n_rows = math.ceil(len(categorical_cols) / n_cols)


fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 5))

axes = axes.flatten()

# --- 各グラフを描画 ---
for i, col in enumerate(categorical_cols):
    ax = axes[i]
    sns.countplot(data=df, x=col, hue='Attrition', palette='viridis', ax=ax)
    ax.set_title(f'"{col}" と "Attrition" の関係')
    ax.tick_params(axis='x', rotation=45)

# --- 余ったグラフを非表示にする ---
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.tight_layout() # レイアウトを自動調整してくれるやつ
plt.show()


In [ ]:
df['OverTime'].unique()

In [ ]:

import math


numerical_cols = [
    'Age', 'DistanceFromHome', 'EnvironmentSatisfaction', 'JobInvolvement',
    'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'NumCompaniesWorked',
    'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel',
    'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
    'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
    'YearsWithCurrManager', 'Incentive', 'RemoteWork'
]


n_cols = 2
n_rows = math.ceil(len(numerical_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 4))

axes = axes.flatten()


for i, col in enumerate(numerical_cols):
    ax = axes[i]
    sns.boxplot(data=df, x='Attrition', y=col, palette='pastel', ax=ax)
    ax.set_title(f'"{col}" と "Attrition" の関係')


for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.tight_layout()
plt.show()

In [ ]:
#可視化

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import math


direct_plot_cols = [

    'BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole',
    'MaritalStatus', 'OverTime', 'HowToEmploy',

    'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel',
    'JobSatisfaction', 'NumCompaniesWorked', 'PerformanceRating',
    'RelationshipSatisfaction', 'StockOptionLevel', 'TrainingTimesLastYear',
    'WorkLifeBalance', 'RemoteWork', 'StressRating'
]


n_cols = 2
n_rows = math.ceil(len(direct_plot_cols) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 5))
axes = axes.flatten()


for i, col in enumerate(direct_plot_cols):
    ax = axes[i]
    crosstab_df = pd.crosstab(df[col], df['Attrition'])
    rate_df = crosstab_df.div(crosstab_df.sum(axis=1), axis=0)
    rate_df.plot(kind='bar', stacked=True, colormap='viridis', ax=ax)


    ax.set_title(f'Attrition Rate by "{col}"', fontsize=14)
    ax.set_ylabel('Proportion')
    ax.tick_params(axis='x', rotation=45)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.legend(title='Attrition')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import math


binning_settings = {
    'Age_Group': {'column': 'Age', 'bins': [18, 30, 40, 50, 61], 'labels': ['18-29', '30-39', '40-49', '50+']},
    'Income_Group': {'column': 'MonthlyIncome', 'bins': [0, 4000, 8000, 12000, float('inf')], 'labels': ['<4k', '4k-8k', '8k-12k', '12k+']},
    'Distance_Group': {'column': 'DistanceFromHome', 'bins': [0, 5, 10, 20, float('inf')], 'labels': ['<5', '5-9', '10-19', '20+']},
    'TotalYears_Group': {'column': 'TotalWorkingYears', 'bins': [0, 10, 20, 30, float('inf')], 'labels': ['<10y', '10-19y', '20-29y', '30y+']},
    'YearsAtCo_Group': {'column': 'YearsAtCompany', 'bins': [0, 5, 10, 15, float('inf')], 'labels': ['<5y', '5-9y', '10-14y', '15y+']},
    'Incentive_Group': {'column': 'Incentive', 'bins': [0, 1000, 3000, 6000, float('inf')], 'labels': ['<1k', '1k-3k', '3k-6k', '6k+']}
}


binned_cols = list(binning_settings.keys())
for new_col, setting in binning_settings.items():
    df[new_col] = pd.cut(df[setting['column']], bins=setting['bins'], labels=setting['labels'], right=False)


n_cols = 2
n_rows = math.ceil(len(binned_cols) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 5))
axes = axes.flatten()


for i, col in enumerate(binned_cols):
    ax = axes[i]
    crosstab_df = pd.crosstab(df[col], df['Attrition'])
    rate_df = crosstab_df.div(crosstab_df.sum(axis=1), axis=0)
    rate_df.plot(kind='bar', stacked=True, colormap='plasma', ax=ax)


    ax.set_title(f'Attrition Rate by "{col}"', fontsize=14)
    ax.set_ylabel('Proportion')
    ax.tick_params(axis='x', rotation=0)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.legend(title='Attrition')


for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.tight_layout()
plt.show()

In [ ]:
# クロス集計を実行
#お試しコード
cross_table = pd.crosstab(df['Attrition'], [df['MonthlyIncome'],df['Incentive']])

print(cross_table)

In [ ]:
# 'JobRole'で絞り込み、'Stress Rating' の各値の出現回数を直接数える
stress_counts_simple = df[df['JobRole'] == 'Sales Representative']['StressRating'].value_counts()

print(stress_counts_simple)

In [ ]:
#  'JobRole' が 'Sales Representative' のデータに絞り込む
sales_reps_df = df[df['JobRole'] == 'Sales Representative']

#  'Stress Rating' ごとにグループ化して、人数を数える
stress_counts = sales_reps_df.groupby('StressRating').size().reset_index(name='Count')

print(stress_counts)
import seaborn as sns
import matplotlib.pyplot as plt

# 上で絞り込んだ 'sales_reps_df' を使う
plt.figure(figsize=(8, 5))
sns.countplot(data=sales_reps_df, x='StressRating', palette='magma')
plt.title('Sales Representative のストレス評価の分布')
plt.xlabel('ストレス評価 (Stress Rating)')
plt.ylabel('人数')
plt.show()

In [ ]:
job_reps_df = df[df['StressRating'] == 5]


stress_counts = job_reps_df.groupby('JobRole').size().reset_index(name='Count')


plt.figure(figsize=(8, 5))
sns.countplot(data=job_reps_df, x='JobRole', palette='magma')
plt.title('jobrole のストレス評価の分布')
plt.xlabel('jobrole')
plt.ylabel('人数')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


fig, axes = plt.subplots(1, 2, figsize=(18, 7))


ax1 = axes[0]
stress_4_df = df[df['StressRating'] == 4]


sns.countplot(data=stress_4_df, x='JobRole', hue='JobRole', palette='viridis', ax=ax1, legend=False)
ax1.set_title('Stress Rating = 4 の職種分布', fontsize=14)
ax1.set_xlabel('JobRole')
ax1.set_ylabel('人数')

ax1.tick_params(axis='x', rotation=45, labelsize=10)



ax2 = axes[1]
stress_5_df = df[df['StressRating'] == 5]


sns.countplot(data=stress_5_df, x='JobRole', hue='JobRole', palette='plasma', ax=ax2, legend=False)
ax2.set_title('Stress Rating = 5 の職種分布', fontsize=14)
ax2.set_xlabel('JobRole')
ax2.set_ylabel('')

ax2.tick_params(axis='x', rotation=45, labelsize=10)


plt.tight_layout()
plt.show()

In [ ]:
df['Attrition'].unique()

In [ ]:
sns.countplot(x='Attrition', data=df)
plt.show()

In [ ]:
df['Attrition'].value_counts()

In [ ]:
counts = df["Attrition"].value_counts()
for label in ["No", "Yes"]:
    print(f"{label},{counts.get(label, 0) / len(df) * 100:.3f}%")


In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# 相関図用の符号化。モデルのカテゴリ処理は後続セルで行います。
encoded_df = raw_df.copy()
for col in encoded_df.select_dtypes(include="object"):
    encoded_df[col] = LabelEncoder().fit_transform(encoded_df[col])


In [ ]:
corr = encoded_df.corr(numeric_only=True)
sns.heatmap(corr, annot=False)
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb

In [ ]:
# 同一の層化分割でベースラインとLightGBMを比較します。
# 旧版の非層化分割・全体LabelEncoderと異なるため旧AUCの再現ではありません。
X = raw_df.drop(columns=['Attrition', 'EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours'])
y = raw_df['Attrition'].map({'No': 0, 'Yes': 1})
if y.isna().any():
    raise ValueError('AttritionはNo/Yesである必要があります。')
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
X_train, X_valid = X_train.copy(), X_valid.copy()
for col in X_train.select_dtypes(include='object'):
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = pd.Categorical(X_valid[col], categories=X_train[col].cat.categories)


In [ ]:
# 公開試行モデル。最終モデル（報告AUC 0.8486）の再現コードではありません。
lgb_train = lgb.Dataset(X_train, y_train)
lgb_eval = lgb.Dataset(X_valid, y_valid, reference=lgb_train)
params = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'num_leaves': 20, 'max_depth': 5, 'min_data_in_leaf': 3,
    'learning_rate': 0.03, 'seed': 0, 'verbosity': -1,
}
model = lgb.train(params, lgb_train, num_boost_round=100,
                  valid_sets=[lgb_eval], callbacks=[lgb.early_stopping(20)])
y_pred = model.predict(X_valid, num_iteration=model.best_iteration)


In [ ]:
lgb.plot_importance(model, importance_type='gain', max_num_features=15, figsize=(10, 8))
plt.title('Feature importance (trial model; not causal effects)')
plt.tight_layout()
plt.show()


In [ ]:
# 別の合成データによるデモを、同じ検証データでの比較に置き換えました。
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
baseline = DummyClassifier(strategy='prior').fit(X_train, y_train)
baseline_pred = baseline.predict_proba(X_valid)[:, 1]
comparison = pd.DataFrame([
    {'Model': name, 'ROC AUC': roc_auc_score(y_valid, pred),
     'Average precision': average_precision_score(y_valid, pred)}
    for name, pred in [('Constant baseline', baseline_pred), ('LightGBM trial', y_pred)]
])
print(comparison.to_string(index=False))
print('検証集合は早期終了にも使用しています。独立テスト性能ではありません。')
